# 06 · 影像分割:逐像素看懂畫面

偵測畫的是方框,但方框裡仍混著背景。**影像分割(segmentation)** 更精細:

> **替每一個像素分類**,精確切出物件的輪廓。分兩種:
> - **語意分割(semantic)**:同類的像素一視同仁(所有「人」是同一片)。
> - **實例分割(instance)**:同類也分開個體(人 1、人 2、人 3 各一片)。

醫療影像、影像去背、自駕的可行駛區域都靠它。這課用 **YOLO 的分割版**做實例分割。

## 1. 安裝

In [ ]:
%pip install -q -U ultralytics

## 2. 載入分割模型,輸出帶遮罩(mask)的結果

`yolov8n-seg.pt` 會替每個物件畫出**輪廓遮罩**,不只是框。

In [ ]:
import matplotlib.pyplot as plt
from ultralytics import YOLO

seg = YOLO("yolov8n-seg.pt")                      # 分割版預訓練權重
result = seg("https://ultralytics.com/images/bus.jpg")[0]

plt.figure(figsize=(8, 6))
plt.imshow(result.plot()[:, :, ::-1])             # 每個物件疊上半透明輪廓遮罩
plt.axis("off"); plt.title("YOLO 實例分割"); plt.show()

## 3. 遮罩其實是一堆 0/1 的張量

每個物件對應一張和影像同尺寸的遮罩:屬於該物件的像素是 1、其餘是 0。

In [ ]:
if result.masks is not None:
    print("遮罩張量形狀:", tuple(result.masks.data.shape))   # (物件數, 高, 寬)
    print("偵測到的物件:", [seg.names[int(b.cls)] for b in result.boxes])
    print("→ 每個物件一張逐像素遮罩,可拿來去背、去填色、算面積。")

## 4. 兩條技術路線

- **實例分割**(這課的 YOLO-seg):物件導向,同類也分個體,適合「數有幾個、各自切開」。
- **語意分割**(如 torchvision 的 `deeplabv3`):像素導向,適合「整片道路 / 整片天空」這種場景理解。

選哪個看任務:要區分個體用實例,要理解整片區域用語意。

In [ ]:
# 語意分割的另一條路(torchvision 內建,概念示意):
# from torchvision.models.segmentation import deeplabv3_resnet50, DeepLabV3_ResNet50_Weights
# w = DeepLabV3_ResNet50_Weights.DEFAULT
# model = deeplabv3_resnet50(weights=w).eval()
# out = model(w.transforms()(img).unsqueeze(0))["out"]      # 每像素一個類別分數
print("實例分割:YOLO-seg；語意分割:deeplabv3。看任務選工具。")

## 小結

- **分割 = 逐像素分類**,比邊界框更精細,切得出輪廓。
- **語意**(同類合一)vs **實例**(同類分個體),依任務選。
- YOLO-seg 開箱即用拿到遮罩張量,可去背、算面積、做後續處理。

下一課:模型說它看到一隻貓——但它**到底在看圖的哪裡**?用 Grad-CAM 打開黑盒子。